# Challenge 1: Structured Agents with FoundryChatClient

## Why Structured Outputs Matter

Most agent tutorials have agents return **free text** — then you regex-parse it and pray.
MAF agents return **Pydantic models** — typed, validated, machine-readable data that can be
routed deterministically through a workflow graph.

In this challenge you'll build agents that:
- Use `FoundryChatClient` (lightweight, local agents — no server-side resource creation)
- Return **Pydantic structured outputs** via `response_format`
- Use MAF's `@tool` decorator with type annotations
- Produce typed data that downstream agents and workflow executors can consume reliably

| Agent | Structured Output | Tools | Purpose |
|-------|------------------|-------|---------|
| **Triage** | `TriageResult` | `check_alert_history`, `get_runbook` | Classify severity, decide response path |
| **Diagnostics** | `DiagnosticsResult` | `get_metrics`, `get_logs`, `check_dependencies` | Find root cause with evidence |
| **Remediation Planner** | `RemediationPlan` | *(none — plans only)* | Determine fix strategy from diagnostics |
| **Verification** | `VerificationResult` | `get_health_status`, `run_smoke_test` | Confirm fix worked |

---

## How This Challenge Works

1. The **Triage Agent** is provided as a complete reference
2. You define the **Pydantic models** for remaining agents' outputs
3. You build the **Diagnostics**, **Remediation Planner**, and **Verification** agents
4. Each agent has a validation cell that asserts on the structured output

> **Key Insight**: When agents return typed data (`TriageResult.severity == "critical"`),
> you can route workflows with Python conditionals — not fragile string matching.

## Setup

In [ ]:
import os
import sys
import json
from typing import Literal

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    get_health_status, run_smoke_test,
)

load_dotenv("../.env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

incident = incidents[0]  # Critical: payment-api OOM
print(f"\U0001f6a8 Incident: {incident['title']}")
print(f"   Service: {incident['service']} | Severity: {incident['severity']}")
print(f"   Type: {incident['incident_type']}")
print(f"   {incident['description']}")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


🚨 Incident: Payment API P99 Latency > 30s
   Service: payment-api | Severity: critical
   Type: high_latency
   payment-api pod-3 OOMKilled. P99 latency spiked to 32s (baseline: 200ms). Connection pool exhausted with 312 requests queued.


---
## Step 1: Define Structured Output Models

These Pydantic models define the **contract** between agents.
When you set `response_format=TriageResult`, MAF forces the LLM to return
valid JSON matching this schema — no parsing errors, no hallucinated fields.

### TriageResult (provided)

In [2]:
class TriageResult(BaseModel):
    """Structured output from the Triage Agent."""
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool = Field(description="Whether this alert pattern has been seen before")
    auto_remediation_allowed: bool = Field(description="Whether the runbook permits auto-fix")
    root_cause_hypothesis: str = Field(description="Initial hypothesis based on alert history")
    recommended_action: str = Field(description="What the Diagnostics Agent should investigate")
    escalation_threshold_minutes: int = Field(description="Minutes before human escalation")

print("\u2705 TriageResult model defined")
print(f"   Fields: {list(TriageResult.model_fields.keys())}")

✅ TriageResult model defined
   Fields: ['severity', 'is_recurring', 'auto_remediation_allowed', 'root_cause_hypothesis', 'recommended_action', 'escalation_threshold_minutes']


### YOUR TURN: Define DiagnosticsResult

The Diagnostics Agent investigates the incident and must return:
- What is the root cause?
- What evidence supports this conclusion?
- Which components are affected?
- How confident is the diagnosis? (0.0 to 1.0)
- What specific fix does it recommend?

In [3]:
class DiagnosticsResult(BaseModel):
    """Structured output from the Diagnostics Agent."""
    root_cause: str = Field(description="The identified root cause of the incident")
    evidence: list[str] = Field(description="List of metrics/logs/data points supporting the diagnosis")
    affected_components: list[str] = Field(description="Pods, services, or components impacted")
    confidence: float = Field(description="Confidence score 0.0 to 1.0", ge=0.0, le=1.0)
    recommended_fix: str = Field(description="Specific remediation action with target details")
    requires_restart: bool = Field(description="Whether the fix requires restarting a pod or service")

print("✅ DiagnosticsResult model defined")
print(f"   Fields: {list(DiagnosticsResult.model_fields.keys())}")


✅ DiagnosticsResult model defined
   Fields: ['root_cause', 'evidence', 'affected_components', 'confidence', 'recommended_fix', 'requires_restart']


### YOUR TURN: Define RemediationPlan

The Remediation Planner reads the diagnostics and produces an execution plan:
- What action to take (restart, scale, flush, toggle flag, escalate)
- Target details (which pod, how many replicas, which flag)
- Risk assessment
- Rollback strategy if the fix fails

In [4]:
class RemediationPlan(BaseModel):
    """Structured output from the Remediation Planner."""
    action: Literal["restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate"] = Field(description="Primary remediation action to take")
    target_service: str = Field(description="Service to act on")
    target_details: str = Field(description="Specific pod ID, replica count, flag name, or cache name")
    risk_level: Literal["low", "medium", "high"] = Field(description="Risk level of this action")
    estimated_downtime_seconds: int = Field(description="Estimated downtime in seconds", ge=0)
    rollback_strategy: str = Field(description="How to undo this action if it fails")
    requires_approval: bool = Field(description="Whether human approval is needed before execution")

print("✅ RemediationPlan model defined")
print(f"   Fields: {list(RemediationPlan.model_fields.keys())}")


✅ RemediationPlan model defined
   Fields: ['action', 'target_service', 'target_details', 'risk_level', 'estimated_downtime_seconds', 'rollback_strategy', 'requires_approval']


### YOUR TURN: Define VerificationResult

In [5]:
class VerificationResult(BaseModel):
    """Structured output from the Verification Agent."""
    service_healthy: bool = Field(description="Whether the service is healthy after remediation")
    tests_passed: int = Field(description="Number of smoke tests that passed", ge=0)
    tests_failed: int = Field(description="Number of smoke tests that failed", ge=0)
    verification_status: Literal["pass", "fail", "degraded"] = Field(description="Overall verification outcome")
    details: str = Field(description="Summary of health check and test results")

print("✅ VerificationResult model defined")
print(f"   Fields: {list(VerificationResult.model_fields.keys())}")


✅ VerificationResult model defined
   Fields: ['service_healthy', 'tests_passed', 'tests_failed', 'verification_status', 'details']


---
## Step 2: Build the Triage Agent (REFERENCE)

Study this carefully — it demonstrates the pattern:
1. `FoundryChatClient` with `AzureCliCredential` (passwordless auth)
2. `response_format=TriageResult` forces structured JSON output
3. Tools passed directly to the `Agent` constructor
4. `agent.run()` returns an `AgentResponse` whose `.text` is valid JSON
5. Validate with `TriageResult.model_validate_json(response.text)`

In [6]:
# Shared client — all agents use the same endpoint
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

triage_agent = Agent(
    client,
    id="triage-agent",
    name="TriageAgent",
    instructions=(
        "You are an incident Triage Agent — the first responder when a production alert fires.\n"
        "\n"
        "When given an alert, you MUST:\n"
        "1. Call check_alert_history to see if this is a recurring pattern\n"
        "2. Call get_runbook with the incident_type to find the operations playbook\n"
        "\n"
        "Based on the results, assess:\n"
        "- Severity (critical/high/medium/low) considering recurrence and blast radius\n"
        "- Whether auto-remediation is safe (from the runbook)\n"
        "- Initial root cause hypothesis\n"
        "- What the Diagnostics Agent should investigate next\n"
        "\n"
        "CONSTRAINTS:\n"
        "- Do NOT attempt to fix anything — only classify and route\n"
        "- Always use both tools before making your assessment\n"
        "- Be specific about what to investigate next"
    ),
    tools=[check_alert_history, get_runbook],
    default_options=OpenAIChatOptions(response_format=TriageResult),
)

print("✅ Triage Agent created")

✅ Triage Agent created


### Run the Triage Agent

In [7]:
# Run the triage agent and validate structured output
triage_response = await triage_agent.run(
    f"New alert fired:\n"
    f"Title: {incident['title']}\n"
    f"Service: {incident['service']}\n"
    f"Type: {incident['incident_type']}\n"
    f"Description: {incident['description']}"
)

# Parse and validate the structured output
triage_result = TriageResult.model_validate_json(triage_response.text)

print("\U0001f3af TRIAGE RESULT (structured):")
print(f"   severity: {triage_result.severity}")
print(f"   is_recurring: {triage_result.is_recurring}")
print(f"   auto_remediation_allowed: {triage_result.auto_remediation_allowed}")
print(f"   root_cause_hypothesis: {triage_result.root_cause_hypothesis}")
print(f"   recommended_action: {triage_result.recommended_action}")
print(f"   escalation_threshold_minutes: {triage_result.escalation_threshold_minutes}")

# THIS is why structured outputs matter — deterministic routing:
print(f"\n\u27a1\ufe0f  Routing decision: severity=={triage_result.severity!r} \u2192 ", end="")
if triage_result.severity == "critical":
    print("FULL PIPELINE (diagnose \u2192 remediate \u2192 verify \u2192 comms)")
elif triage_result.severity == "high":
    print("EXPEDITED PIPELINE (diagnose \u2192 remediate \u2192 verify)")
else:
    print("MONITOR ONLY (log and notify)")

🎯 TRIAGE RESULT (structured):
   severity: high
   is_recurring: True
   auto_remediation_allowed: True
   root_cause_hypothesis: Memory leak in the payment-api connection pool, triggered periodically after batch job execution.
   recommended_action: The Diagnostics Agent should inspect memory metrics of payment-api pod-3, verify the presence of connection pool leaks, and ensure a pod restart addresses the issue effectively.
   escalation_threshold_minutes: 15

➡️  Routing decision: severity=='high' → EXPEDITED PIPELINE (diagnose → remediate → verify)


In [8]:
# ✅ Validation
assert isinstance(triage_result, TriageResult), "Output must be a TriageResult"
assert triage_result.severity in ("critical", "high", "medium", "low")
assert isinstance(triage_result.is_recurring, bool)
assert isinstance(triage_result.auto_remediation_allowed, bool)
assert len(triage_result.root_cause_hypothesis) > 10
assert triage_result.escalation_threshold_minutes > 0
print("✅ Triage Agent validation passed — structured output is correct")

✅ Triage Agent validation passed — structured output is correct


---
## Step 3: Build the Diagnostics Agent (YOUR TURN)

The Diagnostics Agent receives the triage hypothesis and investigates deeper.
It has access to metrics, logs, and dependency health.

**Requirements:**
- Use the shared `client` (already created above)
- Set `response_format=DiagnosticsResult` (your Pydantic model from Step 1)
- Give it these tools: `get_metrics`, `get_logs`, `check_dependencies`
- Instructions should tell it to:
  - Query memory metrics first (based on OOM hypothesis)
  - Check logs for error patterns
  - Verify dependency health
  - Provide confidence score based on evidence quality

In [9]:
diagnostics_agent = Agent(
    client,
    id="diagnostics-agent",
    name="DiagnosticsAgent",
    instructions=(
        "You are an incident Diagnostics Agent. Your job is to find the root cause with evidence.\n"
        "\n"
        "You MUST use all three tools before forming conclusions:\n"
        "1. Call get_metrics with the service name and relevant metric type (memory, latency, error_rate)\n"
        "2. Call get_logs with severity='ERROR' to find error patterns\n"
        "3. Call check_dependencies to check if cascading failures are involved\n"
        "\n"
        "Your output must include:\n"
        "- root_cause: specific and actionable (not vague)\n"
        "- evidence: at least 3 data points from your tool calls\n"
        "- affected_components: exact pod names, service names\n"
        "- confidence: 0.9+ only if all 3 tools agree; 0.7 if 2 agree\n"
        "- recommended_fix: exact command/action (e.g. 'restart payment-api-pod-3')\n"
        "- requires_restart: True if OOM or pod crash is root cause\n"
        "\n"
        "NEVER guess without tool data."
    ),
    tools=[get_metrics, get_logs, check_dependencies],
    default_options=OpenAIChatOptions(response_format=DiagnosticsResult),
)

print("✅ Diagnostics Agent created")


✅ Diagnostics Agent created


### Run & Validate the Diagnostics Agent

In [10]:
# Feed it the triage result — this is how agents chain: typed output → typed input
diag_response = await diagnostics_agent.run(
    f"Investigate this incident based on triage assessment:\n"
    f"Service: {incident['service']}\n"
    f"Triage hypothesis: {triage_result.root_cause_hypothesis}\n"
    f"Recommended investigation: {triage_result.recommended_action}\n"
    f"Original alert: {incident['description']}"
)

diag_result = DiagnosticsResult.model_validate_json(diag_response.text)

print("\U0001f50d DIAGNOSTICS RESULT (structured):")
print(f"   root_cause: {diag_result.root_cause}")
print(f"   confidence: {diag_result.confidence}")
print(f"   affected_components: {diag_result.affected_components}")
print(f"   requires_restart: {diag_result.requires_restart}")
print(f"   evidence:")
for e in diag_result.evidence:
    print(f"     - {e}")
print(f"   recommended_fix: {diag_result.recommended_fix}")

🔍 DIAGNOSTICS RESULT (structured):
   root_cause: The payment-api pod-3 experienced a memory leak, resulting in Out-of-Memory (OOM) errors. This caused connection pool exhaustion as the batch job executed, overwhelming the service.
   confidence: 0.9
   affected_components: ['payment-api-pod-3', 'order-service']
   requires_restart: True
   evidence:
     - Memory utilization for payment-api-pod-3 reached 846 MB out of 1024 MB, triggering an OOMKill.
     - Error logs from payment-api-pod-3 indicated 'java.lang.OutOfMemoryError: Java heap space.'
     - Dependency health showed 'order-service' degrading with high retry volumes, potentially contributing to increased requests to payment-api.
   recommended_fix: Patch the connection pooling logic to address memory leaks and monitor for stability. Immediately restart processes using 'kubectl restart pod payment-api-pod-3'.


In [11]:
# ✅ Validate diagnostics output
assert isinstance(diag_result, DiagnosticsResult), "Output must be DiagnosticsResult"
assert len(diag_result.root_cause) > 10, "Root cause should be descriptive"
assert len(diag_result.evidence) >= 2, "Should have at least 2 pieces of evidence"
assert 0.0 <= diag_result.confidence <= 1.0, "Confidence must be between 0 and 1"
assert len(diag_result.affected_components) >= 1, "Should identify affected components"
assert isinstance(diag_result.requires_restart, bool)
print("✅ Diagnostics Agent validation passed")

✅ Diagnostics Agent validation passed


---
## Step 4: Build the Remediation Planner (YOUR TURN)

This agent **plans** the fix but does NOT execute it. That's critical for HITL:
the plan gets human approval before execution (Challenge 3).

**Requirements:**
- NO tools (it only plans — execution is separate)
- `response_format=RemediationPlan`
- Instructions should tell it to produce a specific, actionable plan
  based on the diagnostics result

In [12]:
planner = Agent(
    client,
    id="remediation-planner",
    name="RemediationPlanner",
    instructions=(
        "You are a Remediation Planner. You produce a safe, specific fix plan based on diagnostics.\n"
        "\n"
        "You have NO execution tools — you only plan. Choose the action based on root cause:\n"
        "- OOM / memory leak → restart_pod (specify exact pod name)\n"
        "- High CPU / traffic spike → scale_service (specify target replica count)\n"
        "- Stale cache → flush_cache (specify cache name)\n"
        "- External dependency down → toggle_feature_flag (specify flag name)\n"
        "- Unclear root cause or low confidence → escalate\n"
        "\n"
        "Rules:\n"
        "- requires_approval=True for any action that affects production traffic\n"
        "- Always include a realistic rollback_strategy\n"
        "- estimated_downtime_seconds should be realistic (pod restart ≈ 30s)\n"
        "- risk_level='high' if action could cause brief outage"
    ),
    default_options=OpenAIChatOptions(response_format=RemediationPlan),
)

print("✅ Remediation Planner created")


✅ Remediation Planner created


### Run & Validate the Remediation Planner

In [13]:
plan_response = await planner.run(
    f"Create a remediation plan based on this diagnosis:\n"
    f"Service: {incident['service']}\n"
    f"Root cause: {diag_result.root_cause}\n"
    f"Confidence: {diag_result.confidence}\n"
    f"Affected components: {diag_result.affected_components}\n"
    f"Requires restart: {diag_result.requires_restart}\n"
    f"Recommended fix: {diag_result.recommended_fix}"
)

plan_result = RemediationPlan.model_validate_json(plan_response.text)

print("\U0001f6e0\ufe0f REMEDIATION PLAN (structured):")
print(f"   action: {plan_result.action}")
print(f"   target_service: {plan_result.target_service}")
print(f"   target_details: {plan_result.target_details}")
print(f"   risk_level: {plan_result.risk_level}")
print(f"   estimated_downtime_seconds: {plan_result.estimated_downtime_seconds}")
print(f"   rollback_strategy: {plan_result.rollback_strategy}")
print(f"   requires_approval: {plan_result.requires_approval}")

🛠️ REMEDIATION PLAN (structured):
   action: restart_pod
   target_service: payment-api
   target_details: pod-3
   risk_level: high
   estimated_downtime_seconds: 30
   rollback_strategy: Revert the recent deployment on payment-api service and reattempt patching.
   requires_approval: True


In [14]:
# ✅ Validate remediation plan
assert isinstance(plan_result, RemediationPlan)
assert plan_result.action in ("restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate")
assert plan_result.risk_level in ("low", "medium", "high")
assert plan_result.estimated_downtime_seconds >= 0
assert len(plan_result.rollback_strategy) > 5, "Rollback strategy must be specified"
assert isinstance(plan_result.requires_approval, bool)
print("✅ Remediation Planner validation passed")

✅ Remediation Planner validation passed


---
## Step 5: Build the Verification Agent (YOUR TURN)

After remediation executes, this agent checks if the fix actually worked.

**Requirements:**
- Tools: `get_health_status`, `run_smoke_test`
- `response_format=VerificationResult`
- Should run health check AND smoke tests, then decide pass/fail/degraded

In [15]:
verifier = Agent(
    client,
    id="verification-agent",
    name="VerificationAgent",
    instructions=(
        "You are a Verification Agent. After remediation, you confirm whether the fix worked.\n"
        "\n"
        "You MUST call both tools:\n"
        "1. Call get_health_status to check the service health endpoints\n"
        "2. Call run_smoke_test to run functional tests against the service\n"
        "\n"
        "Determine verification_status:\n"
        "- 'pass': health check OK AND all smoke tests pass\n"
        "- 'degraded': health check OK BUT some smoke tests fail\n"
        "- 'fail': health check fails (regardless of tests)\n"
        "\n"
        "Count tests_passed and tests_failed accurately from the smoke test results.\n"
        "service_healthy=True only if health check shows all systems operational."
    ),
    tools=[get_health_status, run_smoke_test],
    default_options=OpenAIChatOptions(response_format=VerificationResult),
)

print("✅ Verification Agent created")


✅ Verification Agent created


### Run & Validate

In [16]:
verify_response = await verifier.run(
    f"Verify that the remediation was successful:\n"
    f"Service: {incident['service']}\n"
    f"Action taken: {plan_result.action} on {plan_result.target_details}\n"
    f"Expected outcome: Service healthy, latency back to baseline"
)

verify_result = VerificationResult.model_validate_json(verify_response.text)

print("✅ VERIFICATION RESULT (structured):")
print(f"   service_healthy: {verify_result.service_healthy}")
print(f"   tests_passed: {verify_result.tests_passed}")
print(f"   tests_failed: {verify_result.tests_failed}")
print(f"   verification_status: {verify_result.verification_status}")
print(f"   details: {verify_result.details}")

✅ VERIFICATION RESULT (structured):
   service_healthy: True
   tests_passed: 12
   tests_failed: 0
   verification_status: pass
   details: Both health checks and critical path smoke tests for payment-api passed without any issues. Service response time is within expected baseline at 12ms.


In [17]:
assert isinstance(verify_result, VerificationResult)
assert verify_result.verification_status in ("pass", "fail", "degraded")
assert verify_result.tests_passed >= 0
assert isinstance(verify_result.service_healthy, bool)
print("✅ Verification Agent validation passed")

✅ Verification Agent validation passed


---
## Step 6: Chain All Agents (End-to-End)

Now run the full pipeline manually. Notice how **typed data flows** between agents:
- `TriageResult.severity` → determines routing
- `DiagnosticsResult.root_cause` → feeds remediation planning
- `RemediationPlan.action` → drives execution
- `VerificationResult.verification_status` → determines success/retry

In Challenge 2, you'll wire this into a **graph workflow** with conditional
edges — so the routing happens automatically based on these typed fields.

In [18]:
print("\U0001f3af Full Agent Pipeline Summary")
print("=" * 50)
print(f"\n1. TRIAGE: severity={triage_result.severity}, recurring={triage_result.is_recurring}")
print(f"   \u2192 Hypothesis: {triage_result.root_cause_hypothesis[:60]}...")
print(f"\n2. DIAGNOSTICS: {diag_result.root_cause[:60]}...")
print(f"   \u2192 Confidence: {diag_result.confidence}, Components: {diag_result.affected_components}")
print(f"\n3. PLAN: {plan_result.action} \u2192 {plan_result.target_details}")
print(f"   \u2192 Risk: {plan_result.risk_level}, Downtime: {plan_result.estimated_downtime_seconds}s")
print(f"\n4. VERIFY: {verify_result.verification_status}")
print(f"   \u2192 Tests: {verify_result.tests_passed} passed, {verify_result.tests_failed} failed")
print(f"\n{'='*50}")
print(f"\n\u2728 All data is TYPED \u2014 no string parsing needed.")
print(f"   This is what enables deterministic workflow routing in Challenge 2.")

🎯 Full Agent Pipeline Summary

1. TRIAGE: severity=high, recurring=True
   → Hypothesis: Memory leak in the payment-api connection pool, triggered pe...

2. DIAGNOSTICS: The payment-api pod-3 experienced a memory leak, resulting i...
   → Confidence: 0.9, Components: ['payment-api-pod-3', 'order-service']

3. PLAN: restart_pod → pod-3
   → Risk: high, Downtime: 30s

4. VERIFY: pass
   → Tests: 12 passed, 0 failed


✨ All data is TYPED — no string parsing needed.
   This is what enables deterministic workflow routing in Challenge 2.


---
## ➡️ Next: Challenge 2 — Workflow Graphs & Conditional Routing

You've been manually passing typed outputs between agents.
In Challenge 2, you'll wire these agents into a **MAF workflow graph** with:
- Switch-case routing based on `TriageResult.severity`
- State management with `ctx.set_state()`/`ctx.get_state()`
- The workflow automatically takes different paths for different incidents

[Open Challenge 2 →](../challenge-2/challenge.ipynb)

# Challenge 1: Structured Agents with FoundryChatClient

## Why Structured Outputs Matter

Most agent tutorials have agents return **free text** — then you regex-parse it and pray.
MAF agents return **Pydantic models** — typed, validated, machine-readable data that can be
routed deterministically through a workflow graph.

In this challenge you'll build agents that:
- Use `FoundryChatClient` (lightweight, local agents — no server-side resource creation)
- Return **Pydantic structured outputs** via `response_format`
- Use MAF's `@tool` decorator with type annotations
- Produce typed data that downstream agents and workflow executors can consume reliably

| Agent | Structured Output | Tools | Purpose |
|-------|------------------|-------|---------|
| **Triage** | `TriageResult` | `check_alert_history`, `get_runbook` | Classify severity, decide response path |
| **Diagnostics** | `DiagnosticsResult` | `get_metrics`, `get_logs`, `check_dependencies` | Find root cause with evidence |
| **Remediation Planner** | `RemediationPlan` | *(none — plans only)* | Determine fix strategy from diagnostics |
| **Verification** | `VerificationResult` | `get_health_status`, `run_smoke_test` | Confirm fix worked |

---

## How This Challenge Works

1. The **Triage Agent** is provided as a complete reference
2. You define the **Pydantic models** for remaining agents' outputs
3. You build the **Diagnostics**, **Remediation Planner**, and **Verification** agents
4. Each agent has a validation cell that asserts on the structured output

> **Key Insight**: When agents return typed data (`TriageResult.severity == "critical"`),
> you can route workflows with Python conditionals — not fragile string matching.

## Setup

In [19]:
import os
import sys
import json
from typing import Annotated, Any, Literal

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    get_health_status, run_smoke_test,
)

load_dotenv("../.env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

incident = incidents[0]  # Critical: payment-api OOM
print(f"\U0001f6a8 Incident: {incident['title']}")
print(f"   Service: {incident['service']} | Severity: {incident['severity']}")
print(f"   Type: {incident['incident_type']}")
print(f"   {incident['description']}")

🚨 Incident: Payment API P99 Latency > 30s
   Service: payment-api | Severity: critical
   Type: high_latency
   payment-api pod-3 OOMKilled. P99 latency spiked to 32s (baseline: 200ms). Connection pool exhausted with 312 requests queued.


---
## Step 1: Define Structured Output Models

These Pydantic models define the **contract** between agents.
When you set `response_format=TriageResult`, MAF forces the LLM to return
valid JSON matching this schema — no parsing errors, no hallucinated fields.

### TriageResult (provided)

In [20]:
class TriageResult(BaseModel):
    """Structured output from the Triage Agent."""
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool = Field(description="Whether this alert pattern has been seen before")
    auto_remediation_allowed: bool = Field(description="Whether the runbook permits auto-fix")
    root_cause_hypothesis: str = Field(description="Initial hypothesis based on alert history")
    recommended_action: str = Field(description="What the Diagnostics Agent should investigate")
    escalation_threshold_minutes: int = Field(description="Minutes before human escalation")

print("\u2705 TriageResult model defined")
print(f"   Fields: {list(TriageResult.model_fields.keys())}")

✅ TriageResult model defined
   Fields: ['severity', 'is_recurring', 'auto_remediation_allowed', 'root_cause_hypothesis', 'recommended_action', 'escalation_threshold_minutes']


### YOUR TURN: Define DiagnosticsResult

The Diagnostics Agent investigates the incident and must return:
- What is the root cause?
- What evidence supports this conclusion?
- Which components are affected?
- How confident is the diagnosis? (0.0 to 1.0)
- What specific fix does it recommend?

In [21]:
class DiagnosticsResult(BaseModel):
    """Structured output from the Diagnostics Agent."""
    root_cause: str = Field(description="The identified root cause of the incident")
    evidence: list[str] = Field(description="List of metrics/logs/data points supporting the diagnosis")
    affected_components: list[str] = Field(description="Pods, services, or components impacted")
    confidence: float = Field(description="Confidence score 0.0 to 1.0", ge=0.0, le=1.0)
    recommended_fix: str = Field(description="Specific remediation action with target details")
    requires_restart: bool = Field(description="Whether the fix requires restarting a pod or service")

print("✅ DiagnosticsResult model defined")
print(f"   Fields: {list(DiagnosticsResult.model_fields.keys())}")


✅ DiagnosticsResult model defined
   Fields: ['root_cause', 'evidence', 'affected_components', 'confidence', 'recommended_fix', 'requires_restart']


### YOUR TURN: Define RemediationPlan

The Remediation Planner reads the diagnostics and produces an execution plan:
- What action to take (restart, scale, flush, toggle flag, escalate)
- Target details (which pod, how many replicas, which flag)
- Risk assessment
- Rollback strategy if the fix fails

In [22]:
class RemediationPlan(BaseModel):
    """Structured output from the Remediation Planner."""
    action: Literal["restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate"] = Field(description="Primary remediation action to take")
    target_service: str = Field(description="Service to act on")
    target_details: str = Field(description="Specific pod ID, replica count, flag name, or cache name")
    risk_level: Literal["low", "medium", "high"] = Field(description="Risk level of this action")
    estimated_downtime_seconds: int = Field(description="Estimated downtime in seconds", ge=0)
    rollback_strategy: str = Field(description="How to undo this action if it fails")
    requires_approval: bool = Field(description="Whether human approval is needed before execution")

print("✅ RemediationPlan model defined")
print(f"   Fields: {list(RemediationPlan.model_fields.keys())}")


✅ RemediationPlan model defined
   Fields: ['action', 'target_service', 'target_details', 'risk_level', 'estimated_downtime_seconds', 'rollback_strategy', 'requires_approval']


### YOUR TURN: Define VerificationResult

In [23]:
class VerificationResult(BaseModel):
    """Structured output from the Verification Agent."""
    service_healthy: bool = Field(description="Whether the service is healthy after remediation")
    tests_passed: int = Field(description="Number of smoke tests that passed", ge=0)
    tests_failed: int = Field(description="Number of smoke tests that failed", ge=0)
    verification_status: Literal["pass", "fail", "degraded"] = Field(description="Overall verification outcome")
    details: str = Field(description="Summary of health check and test results")

print("✅ VerificationResult model defined")
print(f"   Fields: {list(VerificationResult.model_fields.keys())}")


✅ VerificationResult model defined
   Fields: ['service_healthy', 'tests_passed', 'tests_failed', 'verification_status', 'details']


---
## Step 2: Build the Triage Agent (REFERENCE)

Study this carefully — it demonstrates the pattern:
1. `FoundryChatClient` with `AzureCliCredential` (passwordless auth)
2. `response_format=TriageResult` forces structured JSON output
3. Tools passed directly to the `Agent` constructor
4. `agent.run()` returns an `AgentResponse` whose `.text` is valid JSON
5. Validate with `TriageResult.model_validate_json(response.text)`

In [24]:
def create_triage_agent() -> Agent:
    """Create the Triage Agent — first responder that classifies incidents."""
    return Agent(
        client=FoundryChatClient(
            project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
            model=os.environ["FOUNDRY_MODEL"],
            credential=AzureCliCredential(),
        ),
        name="TriageAgent",
        instructions=(
            "You are an incident Triage Agent — the first responder when a production alert fires.\n"
            "\n"
            "When given an alert, you MUST:\n"
            "1. Call check_alert_history to see if this is a recurring pattern\n"
            "2. Call get_runbook with the incident_type to find the operations playbook\n"
            "\n"
            "Based on the results, assess:\n"
            "- Severity (critical/high/medium/low) considering recurrence and blast radius\n"
            "- Whether auto-remediation is safe (from the runbook)\n"
            "- Initial root cause hypothesis\n"
            "- What the Diagnostics Agent should investigate next\n"
            "\n"
            "CONSTRAINTS:\n"
            "- Do NOT attempt to fix anything — only classify and route\n"
            "- Always use both tools before making your assessment\n"
            "- Be specific about what to investigate next"
        ),
        tools=[check_alert_history, get_runbook],
        default_options=OpenAIChatOptions[Any](response_format=TriageResult),
    )

print("\u2705 Triage Agent factory defined")

✅ Triage Agent factory defined


### Run the Triage Agent

In [25]:
# Run the triage agent and validate structured output
triage_agent = create_triage_agent()

triage_response = await triage_agent.run(
    f"New alert fired:\n"
    f"Title: {incident['title']}\n"
    f"Service: {incident['service']}\n"
    f"Type: {incident['incident_type']}\n"
    f"Description: {incident['description']}"
)

# Parse and validate the structured output
triage_result = TriageResult.model_validate_json(triage_response.text)

print("\U0001f3af TRIAGE RESULT (structured):")
print(f"   severity: {triage_result.severity}")
print(f"   is_recurring: {triage_result.is_recurring}")
print(f"   auto_remediation_allowed: {triage_result.auto_remediation_allowed}")
print(f"   root_cause_hypothesis: {triage_result.root_cause_hypothesis}")
print(f"   recommended_action: {triage_result.recommended_action}")
print(f"   escalation_threshold_minutes: {triage_result.escalation_threshold_minutes}")

# THIS is why structured outputs matter — deterministic routing:
print(f"\n\u27a1\ufe0f  Routing decision: severity=={triage_result.severity!r} → ", end="")
if triage_result.severity == "critical":
    print("FULL PIPELINE (diagnose → remediate → verify → comms)")
elif triage_result.severity == "high":
    print("EXPEDITED PIPELINE (diagnose → remediate → verify)")
else:
    print("MONITOR ONLY (log and notify)")

🎯 TRIAGE RESULT (structured):
   severity: high
   is_recurring: True
   auto_remediation_allowed: True
   root_cause_hypothesis: Recurring memory spike caused by connection pool memory leak during batch jobs.
   recommended_action: Investigate resource usage patterns and anomalies on pod-3 and validate against connection pool utilization. Check batch job timing and logs for correlations.
   escalation_threshold_minutes: 15

➡️  Routing decision: severity=='high' → EXPEDITED PIPELINE (diagnose → remediate → verify)


### \u2705 Validation

In [26]:
# Validate the triage agent produced correct structured output
assert isinstance(triage_result, TriageResult), "Output must be a TriageResult"
assert triage_result.severity in ("critical", "high", "medium", "low")
assert isinstance(triage_result.is_recurring, bool)
assert isinstance(triage_result.auto_remediation_allowed, bool)
assert len(triage_result.root_cause_hypothesis) > 10
assert triage_result.escalation_threshold_minutes > 0
print("\u2705 Triage Agent validation passed — structured output is correct")

✅ Triage Agent validation passed — structured output is correct


---
## Step 3: Build the Diagnostics Agent (YOUR TURN)

The Diagnostics Agent receives the triage hypothesis and investigates deeper.
It has access to metrics, logs, and dependency health.

**Requirements:**
- Use `FoundryChatClient` with `AzureCliCredential`
- Set `response_format=DiagnosticsResult` (your Pydantic model from Step 1)
- Give it these tools: `get_metrics`, `get_logs`, `check_dependencies`
- Instructions should tell it to:
  - Query memory metrics first (based on OOM hypothesis)
  - Check logs for error patterns
  - Verify dependency health
  - Provide confidence score based on evidence quality

In [27]:
def create_diagnostics_agent() -> Agent:
    """Create the Diagnostics Agent — investigates root cause with evidence."""
    return Agent(
        client=FoundryChatClient(
            project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
            model=os.environ["FOUNDRY_MODEL"],
            credential=AzureCliCredential(),
        ),
        name="DiagnosticsAgent",
        instructions=(
            "You are an incident Diagnostics Agent. Your job is to find the root cause with evidence.\n"
            "\n"
            "You MUST use all three tools before forming conclusions:\n"
            "1. Call get_metrics with the service name and relevant metric type (memory, latency, error_rate)\n"
            "2. Call get_logs with severity='ERROR' to find error patterns\n"
            "3. Call check_dependencies to check if cascading failures are involved\n"
            "\n"
            "Your output must include:\n"
            "- root_cause: specific and actionable (not vague)\n"
            "- evidence: at least 3 data points from your tool calls\n"
            "- affected_components: exact pod names, service names\n"
            "- confidence: 0.9+ only if all 3 tools agree; 0.7 if 2 agree\n"
            "- recommended_fix: exact command/action (e.g. 'restart payment-api-pod-3')\n"
            "- requires_restart: True if OOM or pod crash is root cause\n"
            "\n"
            "NEVER guess without tool data."
        ),
        tools=[get_metrics, get_logs, check_dependencies],
        default_options=OpenAIChatOptions[Any](response_format=DiagnosticsResult),
    )

print("✅ Diagnostics Agent factory defined")


✅ Diagnostics Agent factory defined


### Run & Validate the Diagnostics Agent

In [28]:
diagnostics_agent = create_diagnostics_agent()

# Feed it the triage result — this is how agents chain: typed output → typed input
diag_response = await diagnostics_agent.run(
    f"Investigate this incident based on triage assessment:\n"
    f"Service: {incident['service']}\n"
    f"Triage hypothesis: {triage_result.root_cause_hypothesis}\n"
    f"Recommended investigation: {triage_result.recommended_action}\n"
    f"Original alert: {incident['description']}"
)

diag_result = DiagnosticsResult.model_validate_json(diag_response.text)

print("\U0001f50d DIAGNOSTICS RESULT (structured):")
print(f"   root_cause: {diag_result.root_cause}")
print(f"   confidence: {diag_result.confidence}")
print(f"   affected_components: {diag_result.affected_components}")
print(f"   requires_restart: {diag_result.requires_restart}")
print(f"   evidence:")
for e in diag_result.evidence:
    print(f"     - {e}")
print(f"   recommended_fix: {diag_result.recommended_fix}")

🔍 DIAGNOSTICS RESULT (structured):
   root_cause: Recurring memory spike linked to connection pool memory leak in payment-api-pod-3 during batch jobs execution, causing pod to exceed memory limit and crash.
   confidence: 0.9
   affected_components: ['payment-api-pod-3', 'order-service']
   requires_restart: True
   evidence:
     - Memory utilization for payment-api-pod-3 reached 846 MB out of a 2048 MB limit.
     - Logs from payment-api-pod-3 showed 'java.lang.OutOfMemoryError: Java heap space' and 'Connection pool exhausted — 312 requests waiting'.
     - Order-service upstream dependency is experiencing degraded performance, contributing to retries and increased load.
   recommended_fix: Execute the diagnostics script to identify specific memory leaks in connection pool initialization. Increase the memory limit of pod-3 via Kubernetes settings and patch the code managing connection pools to handle large batch jobs efficiently.


In [29]:
# Validate diagnostics output
assert isinstance(diag_result, DiagnosticsResult), "Output must be DiagnosticsResult"
assert len(diag_result.root_cause) > 10, "Root cause should be descriptive"
assert len(diag_result.evidence) >= 2, "Should have at least 2 pieces of evidence"
assert 0.0 <= diag_result.confidence <= 1.0, "Confidence must be between 0 and 1"
assert len(diag_result.affected_components) >= 1, "Should identify affected components"
assert isinstance(diag_result.requires_restart, bool)
print("\u2705 Diagnostics Agent validation passed")

✅ Diagnostics Agent validation passed


---
## Step 4: Build the Remediation Planner (YOUR TURN)

This agent **plans** the fix but does NOT execute it. That's critical for HITL:
the plan gets human approval before execution (Challenge 3).

**Requirements:**
- NO tools (it only plans — execution is separate)
- `response_format=RemediationPlan`
- Instructions should tell it to produce a specific, actionable plan
  based on the diagnostics result

In [30]:
def create_remediation_planner() -> Agent:
    """Create the Remediation Planner — produces fix plan without executing."""
    return Agent(
        client=FoundryChatClient(
            project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
            model=os.environ["FOUNDRY_MODEL"],
            credential=AzureCliCredential(),
        ),
        name="RemediationPlanner",
        instructions=(
            "You are a Remediation Planner. You produce a safe, specific fix plan based on diagnostics.\n"
            "\n"
            "You have NO execution tools — you only plan. Choose the action based on root cause:\n"
            "- OOM / memory leak → restart_pod (specify exact pod name)\n"
            "- High CPU / traffic spike → scale_service (specify target replica count)\n"
            "- Stale cache → flush_cache (specify cache name)\n"
            "- External dependency down → toggle_feature_flag (specify flag name)\n"
            "- Unclear root cause or low confidence → escalate\n"
            "\n"
            "Rules:\n"
            "- requires_approval=True for any action that affects production traffic\n"
            "- Always include a realistic rollback_strategy\n"
            "- estimated_downtime_seconds should be realistic (pod restart ≈ 30s)\n"
            "- risk_level='high' if action could cause brief outage"
        ),
        default_options=OpenAIChatOptions[Any](response_format=RemediationPlan),
    )

print("✅ Remediation Planner factory defined")


✅ Remediation Planner factory defined


### Run & Validate the Remediation Planner

In [31]:
planner = create_remediation_planner()

plan_response = await planner.run(
    f"Create a remediation plan based on this diagnosis:\n"
    f"Service: {incident['service']}\n"
    f"Root cause: {diag_result.root_cause}\n"
    f"Confidence: {diag_result.confidence}\n"
    f"Affected components: {diag_result.affected_components}\n"
    f"Requires restart: {diag_result.requires_restart}\n"
    f"Recommended fix: {diag_result.recommended_fix}"
)

plan_result = RemediationPlan.model_validate_json(plan_response.text)

print("\U0001f6e0\ufe0f REMEDIATION PLAN (structured):")
print(f"   action: {plan_result.action}")
print(f"   target_service: {plan_result.target_service}")
print(f"   target_details: {plan_result.target_details}")
print(f"   risk_level: {plan_result.risk_level}")
print(f"   estimated_downtime_seconds: {plan_result.estimated_downtime_seconds}")
print(f"   rollback_strategy: {plan_result.rollback_strategy}")
print(f"   requires_approval: {plan_result.requires_approval}")

🛠️ REMEDIATION PLAN (structured):
   action: restart_pod
   target_service: payment-api
   target_details: payment-api-pod-3
   risk_level: medium
   estimated_downtime_seconds: 30
   rollback_strategy: Monitor the pod's memory usage after restarting and revert any settings changes if further issues are detected.
   requires_approval: True


In [32]:
# Validate remediation plan
assert isinstance(plan_result, RemediationPlan)
assert plan_result.action in ("restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate")
assert plan_result.risk_level in ("low", "medium", "high")
assert plan_result.estimated_downtime_seconds >= 0
assert len(plan_result.rollback_strategy) > 5, "Rollback strategy must be specified"
assert isinstance(plan_result.requires_approval, bool)
print("\u2705 Remediation Planner validation passed")

✅ Remediation Planner validation passed


---
## Step 5: Build the Verification Agent (YOUR TURN)

After remediation executes, this agent checks if the fix actually worked.

**Requirements:**
- Tools: `get_health_status`, `run_smoke_test`
- `response_format=VerificationResult`
- Should run health check AND smoke tests, then decide pass/fail/degraded

In [33]:
def create_verification_agent() -> Agent:
    """Create the Verification Agent — confirms fix worked."""
    return Agent(
        client=FoundryChatClient(
            project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
            model=os.environ["FOUNDRY_MODEL"],
            credential=AzureCliCredential(),
        ),
        name="VerificationAgent",
        instructions=(
            "You are a Verification Agent. After remediation, you confirm whether the fix worked.\n"
            "\n"
            "You MUST call both tools:\n"
            "1. Call get_health_status to check the service health endpoints\n"
            "2. Call run_smoke_test to run functional tests against the service\n"
            "\n"
            "Determine verification_status:\n"
            "- 'pass': health check OK AND all smoke tests pass\n"
            "- 'degraded': health check OK BUT some smoke tests fail\n"
            "- 'fail': health check fails (regardless of tests)\n"
            "\n"
            "Count tests_passed and tests_failed accurately from the smoke test results.\n"
            "service_healthy=True only if health check shows all systems operational."
        ),
        tools=[get_health_status, run_smoke_test],
        default_options=OpenAIChatOptions[Any](response_format=VerificationResult),
    )

print("✅ Verification Agent factory defined")


✅ Verification Agent factory defined


### Run & Validate

In [34]:
verifier = create_verification_agent()

verify_response = await verifier.run(
    f"Verify that the remediation was successful:\n"
    f"Service: {incident['service']}\n"
    f"Action taken: {plan_result.action} on {plan_result.target_details}\n"
    f"Expected outcome: Service healthy, latency back to baseline"
)

verify_result = VerificationResult.model_validate_json(verify_response.text)

print("\u2705 VERIFICATION RESULT (structured):")
print(f"   service_healthy: {verify_result.service_healthy}")
print(f"   tests_passed: {verify_result.tests_passed}")
print(f"   tests_failed: {verify_result.tests_failed}")
print(f"   verification_status: {verify_result.verification_status}")
print(f"   details: {verify_result.details}")

✅ VERIFICATION RESULT (structured):
   service_healthy: True
   tests_passed: 46
   tests_failed: 1
   verification_status: degraded
   details: The payment-api health check passed, indicating all systems are operational. However, one out of 47 smoke tests failed in the full suite, leading to a degraded verification status.


In [35]:
assert isinstance(verify_result, VerificationResult)
assert verify_result.verification_status in ("pass", "fail", "degraded")
assert verify_result.tests_passed >= 0
assert isinstance(verify_result.service_healthy, bool)
print("\u2705 Verification Agent validation passed")

✅ Verification Agent validation passed


---
## Step 6: Chain All Agents (End-to-End)

Now run the full pipeline manually. Notice how **typed data flows** between agents:
- `TriageResult.severity` → determines routing
- `DiagnosticsResult.root_cause` → feeds remediation planning
- `RemediationPlan.action` → drives execution
- `VerificationResult.verification_status` → determines success/retry

In Challenge 2, you'll wire this into a **graph workflow** with conditional
edges — so the routing happens automatically based on these typed fields.

In [36]:
print("\U0001f3af Full Agent Pipeline Summary")
print("=" * 50)
print(f"\n1. TRIAGE: severity={triage_result.severity}, recurring={triage_result.is_recurring}")
print(f"   → Hypothesis: {triage_result.root_cause_hypothesis[:60]}...")
print(f"\n2. DIAGNOSTICS: {diag_result.root_cause[:60]}...")
print(f"   → Confidence: {diag_result.confidence}, Components: {diag_result.affected_components}")
print(f"\n3. PLAN: {plan_result.action} → {plan_result.target_details}")
print(f"   → Risk: {plan_result.risk_level}, Downtime: {plan_result.estimated_downtime_seconds}s")
print(f"\n4. VERIFY: {verify_result.verification_status}")
print(f"   → Tests: {verify_result.tests_passed} passed, {verify_result.tests_failed} failed")
print(f"\n{'='*50}")
print(f"\n\u2728 All data is TYPED — no string parsing needed.")
print(f"   This is what enables deterministic workflow routing in Challenge 2.")

🎯 Full Agent Pipeline Summary

1. TRIAGE: severity=high, recurring=True
   → Hypothesis: Recurring memory spike caused by connection pool memory leak...

2. DIAGNOSTICS: Recurring memory spike linked to connection pool memory leak...
   → Confidence: 0.9, Components: ['payment-api-pod-3', 'order-service']

3. PLAN: restart_pod → payment-api-pod-3
   → Risk: medium, Downtime: 30s

4. VERIFY: degraded
   → Tests: 46 passed, 1 failed


✨ All data is TYPED — no string parsing needed.
   This is what enables deterministic workflow routing in Challenge 2.


---
## \U0001f4aa Stretch Goal: Run on Incident #2

Run the same pipeline on the order-service cascading failure (`incidents[1]`).
Observe how agents produce **different structured outputs** for a different scenario:
- Different severity assessment
- Different root cause
- Different remediation action (scale instead of restart)

This proves the agents generalize — the same code handles different incidents.

In [ ]:
# Try with incident #2 (order-service cascading failure)
# incident2 = incidents[1]
# ... run triage, diagnostics, planner, verifier on incident2

---
## \u27a1\ufe0f Next: Challenge 2 — Workflow Graphs & Conditional Routing

You've been manually passing typed outputs between agents.
In Challenge 2, you'll wire these agents into a **MAF workflow graph** with:
- Switch-case routing based on `TriageResult.severity`
- Parallel fan-out for diagnostics
- State management with `ctx.set_state()`/`ctx.get_state()`
- The workflow automatically takes different paths for different incidents

[Open Challenge 2 →](../challenge-2/challenge.ipynb)

# Challenge 1: Specialized Agents with Tools

## What We're Building

Instead of one agent doing everything badly, we'll create **5 specialized agents** that each:
- Have a focused responsibility
- Own specific infrastructure tools
- Produce structured outputs for the next agent

| Agent | Role | Tools |
|-------|------|-------|
| **Triage** | Classify, check history, get runbook | `check_alert_history`, `get_runbook` |
| **Diagnostics** | Find root cause using metrics/logs | `get_metrics`, `get_logs`, `check_dependencies` |
| **Remediation** | Execute the fix | `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`, `escalate_to_human` |
| **Verification** | Confirm fix worked | `get_health_status`, `run_smoke_test` |
| **Communications** | Notify team, create tickets | `post_to_slack`, `create_incident_ticket`, `update_status_page` |

---

## How This Challenge Works

1. The **Triage Agent** is provided as a complete reference
2. You build the remaining **4 agents** yourself
3. Each agent has a validation cell to check your work

> **Tip**: Read the Triage Agent carefully — it shows the pattern for all agents.

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

alert = incidents[0]  # payment-api high latency
print(f"\U0001f534 Working with: {alert['title']}")
print(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"   Description: {alert['description']}")

---
## Agent 1: Triage Agent (REFERENCE — provided for you)

Study this agent. It demonstrates the pattern:
1. Clear role in `instructions`
2. Tool usage guidance (which tool, when)
3. Output format specification
4. Constraints (what NOT to do)

### Expected Output
```
TRIAGE SUMMARY
==============
Severity: Critical
Recurring: Yes — 3 similar alerts in last 7 days
Runbook: high_latency playbook found
Auto-remediation: Allowed
Recommended action: Proceed to diagnostics → likely memory leak based on history
```

In [ ]:
async def build_triage_agent():
    """REFERENCE IMPLEMENTATION — study this pattern, then build the next 4 agents."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        triage_agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent — the first responder when an alert fires.

When given an alert, you MUST:
1. Call check_alert_history with the service name to see if this is recurring
2. Call get_runbook with the incident type (e.g., "high_latency") to find the playbook

Then produce a TRIAGE SUMMARY with:
- Severity assessment (critical/high/medium/low)
- Whether this is a recurring issue (from alert history)
- Whether automated remediation is allowed (from the runbook)
- Recommended next steps for the Diagnostics Agent

CONSTRAINTS:
- Do NOT guess at root causes — that's the Diagnostics Agent's job
- Do NOT suggest fixes — that's the Remediation Agent's job
- ONLY use data from your tools, never make up information""",
            tools=[check_alert_history, get_runbook],
        )

        result = await triage_agent.run(
            f"New alert fired:\n"
            f"Title: {alert['title']}\n"
            f"Service: {alert['service']}\n"
            f"Severity: {alert['severity']}\n"
            f"Description: {alert['description']}\n\n"
            f"Triage this incident now."
        )
        print("\n\U0001f4cb TRIAGE RESULT:\n")
        print(result.text)
        return result.text

triage_output = await build_triage_agent()

---
## Agent 2: Diagnostics Agent (YOUR TURN)

The Diagnostics Agent investigates the root cause using monitoring data.

### Tools Available
| Tool | What it returns |
|------|----------------|
| `get_metrics(service_name)` | CPU, memory, latency, error rate per pod |
| `get_logs(service_name)` | Recent error log entries with timestamps |
| `check_dependencies(service_name)` | Health status of upstream/downstream services |

### Requirements
Your agent must:
- Call ALL THREE tools to gather evidence
- Correlate findings (e.g., high memory + OOM in logs = memory leak)
- Output a **specific** root cause with evidence (not generic advice)
- Include the exact pod/component name that's failing

### Expected Output
```
ROOT CAUSE ANALYSIS
===================
Root cause: Memory leak on payment-api-pod-3
Evidence:
  - Metrics: pod-3 memory at 3891MB/4096MB limit, 4 restarts in last hour
  - Logs: OOMKilled errors at 02:38, 02:41
  - Dependencies: order-service seeing connection timeouts to payment-api
Blast radius: order-service affected (cascading errors)
Recommended fix: Restart payment-api-pod-3 to clear memory leak
```

### Your Task
Fill in the `instructions` and `tools` below:

In [ ]:
async def build_diagnostics_agent(triage_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        diagnostics_agent = client.create_agent(
            name="DiagnosticsAgent",
            # ╔══════════════════════════════════════════════════════╗
            # ║  YOUR CODE: Write the instructions                  ║
            # ║  - State the agent's role                           ║
            # ║  - Tell it which tools to call and in what order    ║
            # ║  - Specify the output format (see Expected Output)  ║
            # ║  - Add constraints (evidence-based, no guessing)    ║
            # ╚══════════════════════════════════════════════════════╝
            instructions="""
TODO: Write your instructions here
""",
            # ╔══════════════════════════════════════════════════════╗
            # ║  YOUR CODE: Provide the correct tools list          ║
            # ║  Which 3 tools does the diagnostics agent need?     ║
            # ╚══════════════════════════════════════════════════════╝
            tools=[],  # TODO: Add the right tools
        )

        result = await diagnostics_agent.run(
            f"Investigate this incident. Triage summary:\n{triage_context}\n\n"
            f"Service: {alert['service']}\n"
            f"Description: {alert['description']}\n\n"
            f"Find the root cause using your monitoring tools."
        )
        print("\n\U0001f50d DIAGNOSTICS RESULT:\n")
        print(result.text)
        return result.text

diagnostics_output = await build_diagnostics_agent(triage_output)

In [ ]:
# ✅ VALIDATION: Check your diagnostics agent worked
assert diagnostics_output, "Diagnostics agent returned empty output!"
output_lower = diagnostics_output.lower()

checks = {
    "Identified pod-3": "pod-3" in output_lower or "pod3" in output_lower,
    "Found memory issue": "memory" in output_lower or "oom" in output_lower,
    "Mentioned restart": "restart" in output_lower,
    "Has evidence": any(w in output_lower for w in ["3891", "4096", "oomkilled", "oom"]),
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Diagnostics agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Review your instructions — is the agent calling all 3 tools?")

---
## Agent 3: Remediation Agent (YOUR TURN)

The Remediation Agent takes **action** on infrastructure. This is the "dangerous" agent.

### Tools Available
| Tool | What it does | When to use |
|------|-------------|-------------|
| `restart_pod(service, pod_id)` | Restarts a specific pod | OOM/memory leak |
| `scale_service(service, replicas)` | Changes replica count | High CPU/traffic |
| `flush_cache(service)` | Clears the cache | Stale data issues |
| `toggle_feature_flag(flag_name, enabled)` | Enables/disables a feature | Failover/degradation |
| `escalate_to_human(reason)` | Pages a human engineer | Can't auto-fix safely |

### Requirements
Your agent must:
- Act ONLY on findings from the diagnostics (not guess)
- Use **exact names** from diagnostics (e.g., `pod-3`, not "the problematic pod")
- Escalate if auto-remediation isn't safe
- Report what action was taken and the result

### Expected Output
```
REMEDIATION ACTIONS
===================
Action taken: Restarted payment-api-pod-3
Result: Pod restarted successfully, new pod healthy
Confidence: High — diagnostics clearly showed OOM on pod-3
```

### Your Task
Build the complete agent (instructions + tools + prompt):

In [ ]:
async def build_remediation_agent(diagnostics_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the entire remediation agent              ║
        # ║  - Write instructions (role, tool selection logic, safety)  ║
        # ║  - Choose the right tools                                   ║
        # ║  - Craft the prompt (pass in diagnostics context)           ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        remediation_agent = client.create_agent(
            name="RemediationAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await remediation_agent.run(
            # TODO: Write your prompt here
            # Include the diagnostics findings and service name
            f"TODO: Write your prompt"
        )
        print("\n\U0001f527 REMEDIATION RESULT:\n")
        print(result.text)
        return result.text

remediation_output = await build_remediation_agent(diagnostics_output)

In [ ]:
# ✅ VALIDATION: Check your remediation agent
assert remediation_output, "Remediation agent returned empty output!"
output_lower = remediation_output.lower()

checks = {
    "Took action (restart)": "restart" in output_lower,
    "Targeted pod-3": "pod-3" in output_lower or "pod3" in output_lower,
    "Mentioned result": any(w in output_lower for w in ["success", "completed", "restarted", "healthy"]),
    "Didn't escalate (auto-fix was allowed)": "escalat" not in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Remediation agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Is the agent using exact pod names from diagnostics?")

---
## Agent 4: Verification Agent (YOUR TURN)

After remediation, we need **proof** the fix worked — not just a hopeful "should be fine."

### Tools Available
| Tool | What it returns |
|------|----------------|
| `get_health_status(service)` | Overall health: latency, error rate, pod status |
| `run_smoke_test(service)` | Functional test results: pass/fail for key endpoints |

### Requirements
Your agent must:
- Call BOTH tools (health check AND smoke test)
- Give a clear **verdict**: `RESOLVED` or `NEEDS_ESCALATION`
- Back up the verdict with data

### Expected Output
```
VERIFICATION REPORT
===================
Health Status: All pods healthy, p95 latency 180ms (within threshold)
Smoke Tests: 5/5 passing (checkout, refund, balance, webhook, auth)

VERDICT: RESOLVED
```

### Your Task

In [ ]:
async def build_verification_agent(remediation_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the verification agent                    ║
        # ║  - Instructions: Check health + run tests, give verdict     ║
        # ║  - Tools: Which 2 tools verify the fix?                     ║
        # ║  - Prompt: Tell it what was fixed so it knows what to check ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        verification_agent = client.create_agent(
            name="VerificationAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await verification_agent.run(
            # TODO: Write your prompt
            f"TODO: Write your prompt"
        )
        print("\n\u2705 VERIFICATION RESULT:\n")
        print(result.text)
        return result.text

verification_output = await build_verification_agent(remediation_output)

In [ ]:
# ✅ VALIDATION: Check your verification agent
assert verification_output, "Verification agent returned empty output!"
output_lower = verification_output.lower()

checks = {
    "Has a verdict": "resolved" in output_lower or "escalat" in output_lower,
    "Checked health": any(w in output_lower for w in ["health", "latency", "pod"]),
    "Ran smoke tests": any(w in output_lower for w in ["smoke", "test", "pass"]),
    "Verdict is RESOLVED (fix should work)": "resolved" in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Verification agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Is the agent calling BOTH health check and smoke test?")

---
## Agent 5: Communications Agent (YOUR TURN)

The final agent handles all human notifications. It doesn't fix anything — it reports.

### Tools Available
| Tool | What it does |
|------|-------------|
| `post_to_slack(channel, message)` | Posts to a Slack channel |
| `create_incident_ticket(title, description, severity)` | Creates a JIRA/PagerDuty ticket |
| `update_status_page(service, status)` | Updates the public status page |

### Requirements
Your agent must call ALL THREE tools:
1. Slack: Brief summary (what happened, root cause, fix, status) — max 4 lines
2. Ticket: Full timeline for post-mortem
3. Status page: Updated to 'operational'

### Expected Output
```
COMMUNICATIONS SENT
===================
Slack (#incidents): Posted 4-line summary
Ticket: INC-2024-001 created with full timeline
Status page: payment-api → operational
```

### Your Task

In [ ]:
async def build_comms_agent(triage_ctx, diagnostics_ctx, remediation_ctx, verification_ctx):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the communications agent                  ║
        # ║  - Instructions: Notify via all 3 channels                  ║
        # ║  - Tools: All 3 communication tools                         ║
        # ║  - Prompt: Pass the full incident timeline                  ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        comms_agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await comms_agent.run(
            # TODO: Write your prompt — include ALL context from the pipeline
            f"TODO: Write your prompt"
        )
        print("\n\U0001f4e2 COMMUNICATIONS RESULT:\n")
        print(result.text)
        return result.text

comms_output = await build_comms_agent(triage_output, diagnostics_output, remediation_output, verification_output)

In [ ]:
# ✅ VALIDATION: Check your communications agent
assert comms_output, "Communications agent returned empty output!"
output_lower = comms_output.lower()

checks = {
    "Posted to Slack": "slack" in output_lower or "#incidents" in output_lower or "posted" in output_lower,
    "Created ticket": "ticket" in output_lower or "inc-" in output_lower or "created" in output_lower,
    "Updated status page": "status" in output_lower and "operational" in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Communications agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Did the agent call all 3 communication tools?")

---
## Full Pipeline Summary

Run this cell to see the complete incident response flow:

In [ ]:
print("\n" + "="*60)
print("\U0001f3c1 FULL INCIDENT RESPONSE — PAYMENT-API")
print("="*60)
print(f"\n\U0001f4cb TRIAGE:\n{triage_output[:300]}")
print(f"\n\U0001f50d DIAGNOSTICS:\n{diagnostics_output[:300]}")
print(f"\n\U0001f527 REMEDIATION:\n{remediation_output[:300]}")
print(f"\n\u2705 VERIFICATION:\n{verification_output[:300]}")
print(f"\n\U0001f4e2 COMMUNICATIONS:\n{comms_output[:300]}")
print("\n" + "="*60)
print("\U0001f389 Incident resolved by 5 specialized agents!")
print("="*60)

---
## \U0001f680 Stretch Goal: Try Incident #3

Run your agents on the notification-service incident. The diagnostics should find
a DIFFERENT root cause (rate limiting, not OOM) and the remediation agent should
take a DIFFERENT action (toggle feature flag, not restart pod).

This proves your agents are reasoning, not just pattern-matching.

In [ ]:
# STRETCH: Run on incident #3 — notification-service
alert = incidents[2]  # Switch to notification-service
print(f"\U0001f534 NEW INCIDENT: {alert['title']}")
print(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"   Description: {alert['description']}")
print("\nRunning full pipeline...\n")

triage_2 = await build_triage_agent()
diag_2 = await build_diagnostics_agent(triage_2)
remed_2 = await build_remediation_agent(diag_2)
verify_2 = await build_verification_agent(remed_2)
comms_2 = await build_comms_agent(triage_2, diag_2, remed_2, verify_2)

# Check that it made different decisions
print("\n" + "="*60)
print("\U0001f9ea DID THE AGENT MAKE DIFFERENT DECISIONS?")
print("="*60)
checks = {
    "NOT restart (different root cause)": "restart" not in remed_2.lower() or "feature" in remed_2.lower(),
    "Toggle feature flag OR different action": any(w in remed_2.lower() for w in ["feature_flag", "toggle", "backup", "failover"]),
}
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

---
## What You Achieved

| What | Single Agent (Basic Approach) | Your Agents (Challenge 1) |
|------|--------------------------|---------------------------|
| Tools | None | 15 real infrastructure APIs |
| Diagnosis | Guesses | Evidence-based (metrics + logs) |
| Fix | Suggests | Executes (restart, scale, toggle) |
| Verification | None | Health checks + smoke tests |
| Communication | None | Slack + ticket + status page |
| Different incidents | Same generic response | Different actions per root cause |

**But there's a problem:** You're manually passing outputs between agents.
What if verification fails? What if you want this to run without you at 3 AM?

---

## \u27a1\ufe0f Next: Challenge 2 — Workflow Orchestration

Wire these agents into an automated pipeline with conditional routing and retry logic.

# 🛠️ Step 2: Specialized Agents with Tools

## What we're building

Instead of one agent doing everything badly, we'll create **specialized agents** that each:
- Have a focused responsibility
- Own specific tools (infrastructure APIs)
- Produce structured outputs for the next agent

## Our Agents

| Agent | Role | Tools |
|-------|------|-------|
| **Triage Agent** | Classify severity, check history, get runbook | `check_alert_history`, `get_runbook` |
| **Diagnostics Agent** | Find root cause using metrics and logs | `get_metrics`, `get_logs`, `check_dependencies` |
| **Remediation Agent** | Execute the fix | `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag` |
| **Verification Agent** | Confirm the fix worked | `get_health_status`, `run_smoke_test` |
| **Comms Agent** | Notify team, create tickets | `post_to_slack`, `create_incident_ticket`, `update_status_page` |

---

## 💡 Key Concept: Task Decomposition

Each agent is an **expert** at one thing. This means:
- Simpler instructions (less confusion)
- Focused tool sets (less chance of misuse)
- Testable in isolation
- Replaceable/upgradable independently

In [ ]:
import os
import json
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

# Import our mock infrastructure tools
import sys
sys.path.insert(0, "..")

from tools.mock_infra import (
    # Triage tools
    check_alert_history,
    get_runbook,
    # Diagnostics tools
    get_metrics,
    get_logs,
    check_dependencies,
    # Remediation tools
    restart_pod,
    scale_service,
    flush_cache,
    toggle_feature_flag,
    # Verification tools
    get_health_status,
    run_smoke_test,
    # Communications tools
    post_to_slack,
    create_incident_ticket,
    update_status_page,
load_dotenv("../.env")
)

with open("../data/incidents.json") as f:

# Load incident data
with open("data/incidents.json") as f:
    incidents = json.load(f)


alert = incidents[0]  # payment-api high latencyprint(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"🔴 Working with alert: {alert['title']}")

## Agent 1: Triage Agent

The Triage Agent is the first responder. Its job:
1. Check if this is a known/recurring issue
2. Look up the relevant runbook
3. Classify whether automated remediation is possible

### 🎯 YOUR TASK
Fill in the `instructions` for the Triage Agent. It should:
- Use `check_alert_history` to see if this happened before
- Use `get_runbook` to find the remediation playbook
- Output a clear triage summary with: severity, whether it's recurring, and recommended action

In [ ]:
async def build_triage_agent():
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Triage Agent
        # Hint: Tell it its role, what tools it has, and what output format you expect
        triage_instructions = """
You are an incident Triage Agent. Your job is to be the first responder when an alert fires.

When given an alert:
1. Use check_alert_history to see if this service has had similar issues recently
2. Use get_runbook to look up the standard operating procedure for this type of incident
3. Produce a triage summary with:
   - Severity assessment (critical/high/medium/low)
   - Whether this is a recurring issue
   - Whether automated remediation is allowed (from the runbook)
   - Recommended next steps for the Diagnostics Agent

Be concise and factual. Do not guess at root causes — that's the Diagnostics Agent's job.
"""

        triage_agent = client.create_agent(
            name="TriageAgent",
            instructions=triage_instructions,
            tools=[check_alert_history, get_runbook],
        )
        
        # Run the triage agent on our alert
        triage_prompt = f"""New alert received:
Title: {alert['title']}
Service: {alert['service']}
Severity: {alert['severity']}
Description: {alert['description']}

Triage this incident."""

        result = await triage_agent.run(triage_prompt)
        print("\n📋 TRIAGE RESULT:\n")
        print(result.text)
        return result.text

triage_output = await build_triage_agent()

## Agent 2: Diagnostics Agent

The Diagnostics Agent digs into the technical details. Its job:
1. Pull real metrics (CPU, memory, latency, error rates)
2. Analyze logs for error patterns
3. Check dependency health
4. Identify the root cause

### 🎯 YOUR TASK
Fill in the `instructions` for the Diagnostics Agent. It should:
- Use its tools to gather evidence (not guess!)
- Correlate findings across metrics, logs, and dependencies
- Output a root cause analysis with specific evidence

In [ ]:
async def build_diagnostics_agent(triage_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Diagnostics Agent
        # Hint: It should systematically gather evidence using all 3 tools
        diagnostics_instructions = """
You are an incident Diagnostics Agent. Your job is to identify the root cause of an incident.

You have access to real monitoring data. When investigating:
1. Use get_metrics to check CPU, memory, latency, and error rates for the affected service
2. Use get_logs to find error messages and patterns
3. Use check_dependencies to see if upstream/downstream services are healthy

Your output should be a root cause analysis with:
- What is failing (specific pod, specific component)
- Why it's failing (evidence from metrics/logs)
- What's the blast radius (which other services are affected)
- Recommended remediation action with specific parameters
  (e.g., "restart payment-api-pod-3" not just "restart a pod")

IMPORTANT: Base your analysis ONLY on data from your tools. Do not speculate.
"""

        diagnostics_agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions=diagnostics_instructions,
            tools=[get_metrics, get_logs, check_dependencies],
        )

        # Pass triage context + original alert
        diag_prompt = f"""Investigate this incident. The Triage Agent has already assessed it:

--- TRIAGE SUMMARY ---
{triage_context}

--- ORIGINAL ALERT ---
Service: {alert['service']}
Description: {alert['description']}

Run diagnostics and identify the root cause."""

        result = await diagnostics_agent.run(diag_prompt)
        print("\n🔍 DIAGNOSTICS RESULT:\n")
        print(result.text)
        return result.text

diagnostics_output = await build_diagnostics_agent(triage_output)

## Agent 3: Remediation Agent

The Remediation Agent takes action. It has the "dangerous" tools that actually change infrastructure.

### 🎯 YOUR TASK
Fill in the `instructions` for the Remediation Agent. Key considerations:
- It should ONLY act based on the diagnostics findings (not guess)
- It should check the runbook's `auto_remediation_allowed` flag
- If auto-remediation isn't allowed, it should call `escalate_to_human` instead

In [ ]:
async def build_remediation_agent(diagnostics_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Remediation Agent
        remediation_instructions = """
You are an incident Remediation Agent. Your job is to execute fixes based on the diagnosis.

You have access to infrastructure tools:
- restart_pod: Restart a specific pod (use when OOM/memory leak detected)
- scale_service: Scale to more replicas (use when CPU is high or traffic spike)
- flush_cache: Flush a cache (use when stale data suspected)
- toggle_feature_flag: Enable/disable features (use for graceful degradation)
- escalate_to_human: Page a human when auto-fix isn't possible

Rules:
1. ONLY take actions that the diagnostics directly support
2. If the runbook says auto_remediation_allowed=False, call escalate_to_human
3. Be specific: use exact pod names, exact service names, exact flag names from the diagnostics
4. Report what actions you took and their results

Do NOT take multiple unrelated actions. Fix the root cause identified by diagnostics.
"""

        remediation_agent = client.create_agent(
            name="RemediationAgent",
            instructions=remediation_instructions,
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )

        remediation_prompt = f"""Execute remediation based on this diagnosis:

--- DIAGNOSTICS FINDINGS ---
{diagnostics_context}

--- SERVICE ---
{alert['service']}

Take the appropriate remediation action now."""

        result = await remediation_agent.run(remediation_prompt)
        print("\n🔧 REMEDIATION RESULT:\n")
        print(result.text)
        return result.text

remediation_output = await build_remediation_agent(diagnostics_output)

## Agent 4: Verification Agent

After remediation, we need to verify the fix actually worked.

### 🎯 YOUR TASK
Create the Verification Agent. It should:
- Check service health after the fix
- Run smoke tests
- Report whether the incident is resolved or needs escalation

In [ ]:
async def build_verification_agent(remediation_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write the instructions and create the agent
        verification_instructions = """
You are a Verification Agent. After remediation actions are taken, you verify the fix worked.

Steps:
1. Use get_health_status to check if the service is now healthy
2. Use run_smoke_test to run functional tests against the service
3. Report your findings:
   - Is the service healthy? (latency, error rate, resource usage)
   - Are smoke tests passing?
   - Final verdict: RESOLVED or NEEDS_ESCALATION

If all checks pass, mark as RESOLVED.
If any checks fail, mark as NEEDS_ESCALATION with details.
"""

        verification_agent = client.create_agent(
            name="VerificationAgent",
            instructions=verification_instructions,
            tools=[get_health_status, run_smoke_test],
        )

        verify_prompt = f"""The following remediation was performed:

--- REMEDIATION ACTIONS ---
{remediation_context}

--- SERVICE ---
{alert['service']}

Verify that the service is now healthy and the incident is resolved."""

        result = await verification_agent.run(verify_prompt)
        print("\n✅ VERIFICATION RESULT:\n")
        print(result.text)
        return result.text

verification_output = await build_verification_agent(remediation_output)

## Agent 5: Communications Agent

The final agent handles all human communication.

### 🎯 YOUR TASK
Create the Comms Agent. It should:
- Post a resolution summary to Slack
- Create a post-incident ticket
- Update the status page

In [ ]:
async def build_comms_agent(triage_ctx: str, diagnostics_ctx: str, remediation_ctx: str, verification_ctx: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Communications Agent
        comms_instructions = """
You are the Communications Agent. After an incident is resolved, you handle all notifications.

Your responsibilities:
1. Post a concise incident summary to Slack (#incidents channel)
2. Create a post-incident ticket with full details for the post-mortem
3. Update the status page to 'operational' if resolved

Your Slack message should be brief and include:
- What happened (1 sentence)
- Root cause (1 sentence)
- Fix applied (1 sentence)
- Current status (resolved/monitoring)

The incident ticket should have the full timeline and details.
"""

        comms_agent = client.create_agent(
            name="CommunicationsAgent",
            instructions=comms_instructions,
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )

        comms_prompt = f"""The incident has been resolved. Notify the team.

--- FULL INCIDENT TIMELINE ---

TRIAGE:
{triage_ctx}

DIAGNOSTICS:
{diagnostics_ctx}

REMEDIATION:
{remediation_ctx}

VERIFICATION:
{verification_ctx}

--- SERVICE ---
{alert['service']}

Post to Slack, create an incident ticket, and update the status page."""

        result = await comms_agent.run(comms_prompt)
        print("\n📢 COMMUNICATIONS RESULT:\n")
        print(result.text)
        return result.text

comms_output = await build_comms_agent(triage_output, diagnostics_output, remediation_output, verification_output)

## 🎉 What We've Built

We now have 5 specialized agents, each with focused tools:

```
Alert ──► Triage ──► Diagnostics ──► Remediation ──► Verification ──► Comms
          (2 tools)   (3 tools)       (5 tools)       (2 tools)       (3 tools)
```

**But there's a problem...**

We're manually passing outputs between agents. What if:
- The verification fails? We need to loop back to remediation.
- We want to run this without human intervention?
- We want to handle different incident types with different routing?

That's where **workflow orchestration** comes in.

---

## ➡️ Next: Step 3 — Workflow Orchestration

In the next notebook, we'll wire these agents into an automated `WorkflowBuilder` that:
- Chains agents together automatically
- Routes based on conditions (fix worked → comms, fix failed → retry/escalate)
- Runs the entire pipeline with one call